In [ ]:
import pandas as pd
from rdkit import Chem

# ==========================================
# 1. データの読み込み
# ==========================================
csv_file = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\merged_basic_descriptors.csv"
df = pd.read_csv(csv_file)

# 列名の前後の空白を削除
df.columns = df.columns.str.strip()

# SMILES列の表記揺れに対応（大文字小文字どちらでもOKにする）
smiles_col = 'smiles' if 'smiles' in df.columns else 'SMILES'

if smiles_col not in df.columns:
    print(f"エラー: CSVファイルに '{smiles_col}' 列が見つかりません。")
    print("実際の列名一覧:", df.columns.tolist())
    exit()

# ==========================================
# 2. エラーSMILESの判定と抽出
# ==========================================
error_records = []

print("SMILESの文法チェックを開始します...")

for index, row in df.iterrows():
    smiles_str = str(row[smiles_col]).strip()
    
    # 欠損値のスキップ
    if smiles_str == 'nan' or not smiles_str:
        continue
        
    # RDKitで分子オブジェクトに変換（読めるかテスト）
    mol = Chem.MolFromSmiles(smiles_str)
    
    # 読み込めなかった場合（Noneが返ってきた場合）はエラーリストに追加
    if mol is None:
        error_records.append({
            'CSV_Row_Number': index + 2, # Excel等の行番号に合わせるため (+2)
            'material_id': row.get('material_id', 'N/A'),
            'name': row.get('name', 'N/A'),
            'invalid_smiles': smiles_str
        })

# ==========================================
# 3. 結果の出力と保存
# ==========================================
df_errors = pd.DataFrame(error_records)

print("\n" + "="*40)
print(f"  🔍 チェック完了: 合計 {len(df_errors)} 件のエラーが見つかりました")
print("="*40)

if len(df_errors) > 0:
    print(df_errors.to_string(index=False))
    
    # エラーリストを新しいCSVとして保存
    output_filename = r"../../data/interim/smiles_error_list.csv"
    df_errors.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n📁 エラーの一覧を '{output_filename}' に保存しました。")
else:
    print("🎉 素晴らしい！すべてのSMILESが正常に読み込めました。")

SMILESの文法チェックを開始します...


[00:50:56] SMILES Parse Error: syntax error while parsing: OC[C@H]1OC(O)C@@HC@@H[C@H]1O
[00:50:56] SMILES Parse Error: check for mistakes around position 15:
[00:50:56] OC[C@H]1OC(O)C@@HC@@H[C@H]1O
[00:50:56] ~~~~~~~~~~~~~~^
[00:50:56] SMILES Parse Error: Failed parsing SMILES 'OC[C@H]1OC(O)C@@HC@@H[C@H]1O' for input: 'OC[C@H]1OC(O)C@@HC@@H[C@H]1O'
[00:50:56] SMILES Parse Error: syntax error while parsing: OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O
[00:50:56] SMILES Parse Error: check for mistakes around position 16:
[00:50:56] OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O
[00:50:56] ~~~~~~~~~~~~~~~^
[00:50:56] SMILES Parse Error: Failed parsing SMILES 'OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O' for input: 'OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O'
[00:50:56] SMILES Parse Error: syntax error while parsing: OCC@H[C@@H]1OC(=O)C(O)=C1O
[00:50:56] SMILES Parse Error: check for mistakes around position 4:
[00:50:56] OCC@H[C@@H]1OC(=O)C(O)=C1O
[00:50:56] ~~~^
[00:50:56] SMILES Parse Error: Failed parsing SMILES 'OCC@H[C@@H]1OC


  🔍 チェック完了: 合計 185 件のエラーが見つかりました
 CSV_Row_Number  material_id                            name                                                                                                   invalid_smiles
            836          329                 Roselle extract                                                                                     OC[C@H]1OC(O)C@@HC@@H[C@H]1O
            859          329                 Roselle extract                                                                                OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O
            868          329                 Roselle extract                                                                                       OCC@H[C@@H]1OC(=O)C(O)=C1O
            869          329                 Roselle extract                                                                                     OC[C@H]1OC(O)C@@HC@H[C@@H]1O
            873          329                 Roselle extract                                    

In [ ]:
import pandas as pd
from rdkit import Chem

# ==========================================
# 1. データの読み込み
# ==========================================
csv_file = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\merged_basic_descriptors.csv"
df = pd.read_csv(csv_file)

df.columns = df.columns.str.strip()
smiles_col = 'smiles' if 'smiles' in df.columns else 'SMILES'

if smiles_col not in df.columns:
    print(f"エラー: CSVファイルに '{smiles_col}' 列が見つかりません。")
    exit()

# ==========================================
# 2. エラーSMILESの判定と抽出
# ==========================================
error_records = []

# すでに登場したSMILESを記録して「純粋な重複」を自動除外するためのセット
seen_smiles = set()

print("SMILESの文法チェックを開始します（重複・既知の除外対象はスキップ）...")

for index, row in df.iterrows():
    smiles_str = str(row[smiles_col]).strip()
    
    # 欠損値のスキップ
    if smiles_str == 'nan' or not smiles_str:
        continue
        
    # ─── 💡 【修正ポイント：除外フィルター】 ───
    
    # 同じ無効SMILESが何度も出てくる「重複」を除外
    if smiles_str in seen_smiles:
        continue  # すでに数えたのでスキップ
    seen_smiles.add(smiles_str)
        
    # ─────────────────────────────────────────
        
    # RDKitで分子オブジェクトに変換（読めるかテスト）
    mol = Chem.MolFromSmiles(smiles_str)
    
    # 読み込めなかった場合（Noneが返ってきた場合）のみ純粋なエラーとしてカウント
    if mol is None:
        error_records.append({
            'CSV_Row_Number': index + 2,
            'material_id': row.get('material_id', 'N/A'),
            'name': row.get('name', 'N/A'),
            'invalid_smiles': smiles_str
        })

# ==========================================
# 3. 結果の出力と保存
# ==========================================
df_errors = pd.DataFrame(error_records)

print("\n" + "="*40)
print(f" 🔍 チェック完了: 純粋な無効SMILESは合計 {len(df_errors)} 件でした")
print("="*40)

if len(df_errors) > 0:
    print(df_errors.to_string(index=False))
    
    output_filename = r"../../data/interim/pure_smiles_error_list.csv"
    df_errors.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n📁 純粋なエラー一覧を '{output_filename}' に保存しました。")
else:
    print("🎉 素晴らしい！ノイズを除外した結果、無効なSMILESは0件でした。")

SMILESの文法チェックを開始します（重複・既知の除外対象はスキップ）...

 🔍 チェック完了: 純粋な無効SMILESは合計 28 件でした
 CSV_Row_Number  material_id                            name                                                                                                   invalid_smiles
            836          329                 Roselle extract                                                                                     OC[C@H]1OC(O)C@@HC@@H[C@H]1O
            859          329                 Roselle extract                                                                                OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O
            868          329                 Roselle extract                                                                                       OCC@H[C@@H]1OC(=O)C(O)=C1O
            869          329                 Roselle extract                                                                                     OC[C@H]1OC(O)C@@HC@H[C@@H]1O
            873          329                 Roselle ex

[15:45:14] SMILES Parse Error: syntax error while parsing: OC[C@H]1OC(O)C@@HC@@H[C@H]1O
[15:45:14] SMILES Parse Error: check for mistakes around position 15:
[15:45:14] OC[C@H]1OC(O)C@@HC@@H[C@H]1O
[15:45:14] ~~~~~~~~~~~~~~^
[15:45:14] SMILES Parse Error: Failed parsing SMILES 'OC[C@H]1OC(O)C@@HC@@H[C@H]1O' for input: 'OC[C@H]1OC(O)C@@HC@@H[C@H]1O'
[15:45:14] SMILES Parse Error: syntax error while parsing: OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O
[15:45:14] SMILES Parse Error: check for mistakes around position 16:
[15:45:14] OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O
[15:45:14] ~~~~~~~~~~~~~~~^
[15:45:14] SMILES Parse Error: Failed parsing SMILES 'OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O' for input: 'OC(=O)C1OC(=O)C@H[C@H]1C(O)C(=O)O'
[15:45:14] SMILES Parse Error: syntax error while parsing: OCC@H[C@@H]1OC(=O)C(O)=C1O
[15:45:14] SMILES Parse Error: check for mistakes around position 4:
[15:45:14] OCC@H[C@@H]1OC(=O)C(O)=C1O
[15:45:14] ~~~^
[15:45:14] SMILES Parse Error: Failed parsing SMILES 'OCC@H[C@@H]1OC